# Module 4 — Q&A RAG Pipeline (Sakina)

Brief requirement: build a RAG pipeline over the [`Amod/mental_health_counseling_conversations`](https://huggingface.co/datasets/Amod/mental_health_counseling_conversations) dataset (~3.5k counselor Q&A pairs). Approach: hybrid retrieval — BM25 + dense `BAAI/bge-m3` (1024-dim) fused via RRF (k=60), reranked with `BAAI/bge-reranker-v2-m3` to top-3, served from Qdrant collection `sakina_counseling`. Evaluated with Hit@5/MRR and an LLM-judge for faithfulness/answer-relevancy.

## ⚠️ License + ethics note

The Amod dataset has no explicit OSI license tag on the Hub. We embed and index the **counselor responses**, never expose raw chunks verbatim (they're synthesized into a grounded LLM reply), and show a footer disclaimer: *"Sakina is not a substitute for professional care."* **Flag to the repo owner:** clarify redistribution terms with the dataset author before deploying publicly beyond academic submission.

## 1. Setup

In [1]:
import random
from pathlib import Path

random.seed(SEED if "SEED" in dir() else 42)

DEVICE = "cpu"
BF16_OK = False
GPU_ARCH = "none"
ARTIFACTS_DIR = Path("..") / "api" / "app" / "models" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)


class LightningRateLimiter:
    """Sliding-window limiter for Lightning's 15 req/min cap.
    `.acquire(estimated_tokens=...)` accepts (and ignores) the token hint."""

    def __init__(self, max_req_per_min: int = 15, window_s: float = 60.0, safety: float = 0.9):
        import threading
        import time
        from collections import deque

        self._max = int(max_req_per_min * safety)
        self._win = window_s
        self._events: deque[float] = deque()
        self._lock = threading.Lock()
        self._time = time

    def acquire(self, estimated_tokens: int = 0) -> None:
        while True:
            with self._lock:
                now = self._time.monotonic()
                while self._events and self._events[0] < now - self._win:
                    self._events.popleft()
                if len(self._events) + 1 <= self._max:
                    self._events.append(now)
                    return
                wait = max(0.1, self._events[0] + self._win - now + 0.05)
            self._time.sleep(wait)


print(f"local setup: DEVICE={DEVICE}, BF16_OK={BF16_OK}, GPU_ARCH={GPU_ARCH}")
print(f"artifacts dir: {ARTIFACTS_DIR.resolve()}")

local setup: DEVICE=cpu, BF16_OK=False, GPU_ARCH=none
artifacts dir: /home/omargamalelkady/ITI/Courses/NLP/Project/sakina/api/app/models/artifacts


In [2]:
from __future__ import annotations

import json
import os
import pickle
import re
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, PointStruct, VectorParams
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, SentenceTransformer
from tqdm.auto import tqdm

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

ENV_PATH = Path("../api/.env")
if ENV_PATH.exists():
    load_dotenv(ENV_PATH)
    print(f"loaded env from {ENV_PATH.resolve()}")
else:
    print(f"no .env at {ENV_PATH.resolve()} — assuming env vars set externally")

for var in ("QDRANT_URL", "QDRANT_API_KEY", "LIGHTNING_API_KEY", "LIGHTNING_MODEL"):
    val = os.environ.get(var, "")
    print(f"  {var:<22} {'set (***' + val[-4:] + ')' if val else 'MISSING'}")

loaded env from /home/omargamalelkady/ITI/Courses/NLP/Project/sakina/api/.env
  QDRANT_URL             set (***6333)
  QDRANT_API_KEY         set (***5-aU)
  LIGHTNING_API_KEY      set (***7057)
  LIGHTNING_MODEL        set (***120b)


## 2. Load the Amod counseling dataset

In [3]:
ds = load_dataset("Amod/mental_health_counseling_conversations", split="train")
print(ds)
print(f"\ncolumns: {ds.column_names}")
print(f"\nFirst example:")
print(f"  Context : {ds[0]['Context'][:200]}...")
print(f"  Response: {ds[0]['Response'][:200]}...")

Dataset({
    features: ['Context', 'Response'],
    num_rows: 3512
})

columns: ['Context', 'Response']

First example:
  Context : I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.
   I've never tried or contemplated suicide. I...
  Response: If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can ...


## 3. EDA — corpus stats

In [4]:
df = pd.DataFrame(
    {
        "context": ds["Context"],
        "response": ds["Response"],
    }
)
df["ctx_len"] = df["context"].str.split().str.len()
df["resp_len"] = df["response"].str.split().str.len()

print(f"# pairs : {len(df)}")
print(f"\nContext length (words):")
print(df["ctx_len"].describe())
print(f"\nResponse length (words):")
print(df["resp_len"].describe())

# pairs : 3512

Context length (words):
count    3512.000000
mean       55.180809
std        48.275077
min         5.000000
25%        28.000000
50%        46.000000
75%        68.000000
max       526.000000
Name: ctx_len, dtype: float64

Response length (words):
count    3512.000000
mean      177.001993
std       120.744872
min         0.000000
25%        93.000000
50%       144.000000
75%       221.000000
max       939.000000
Name: resp_len, dtype: float64


## 4. Clean + chunk

Embed the **counselor response** (the helpful text); context is kept as metadata. Chunk size 300 words, overlap 50; PII (email/phone/URL) is scrubbed as a precaution.

In [5]:
EMAIL_RE = re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b")
PHONE_RE = re.compile(r"\b(\+?\d[\d\s\-().]{7,}\d)\b")
URL_RE = re.compile(r"https?://\S+|www\.\S+")
WS_RE = re.compile(r"\s+")


def clean(text: str) -> str:
    text = EMAIL_RE.sub("[email]", text)
    text = PHONE_RE.sub("[phone]", text)
    text = URL_RE.sub("[url]", text)
    text = WS_RE.sub(" ", text).strip()
    return text


def word_chunks(text: str, size: int = 300, overlap: int = 50) -> list[str]:
    words = text.split()
    if len(words) <= size:
        return [text]
    chunks, start, step = [], 0, size - overlap
    while start < len(words):
        chunks.append(" ".join(words[start : start + size]))
        if start + size >= len(words):
            break
        start += step
    return chunks


chunks: list[dict] = []
for idx, row in tqdm(df.iterrows(), total=len(df), desc="chunking"):
    response = clean(row["response"])
    context = clean(row["context"])
    if not response or len(response.split()) < 10:
        continue  # drop near-empty responses
    for i, ch in enumerate(word_chunks(response, size=300, overlap=50)):
        chunks.append(
            {
                "id": f"{idx}-{i}",
                "text": ch,
                "context": context,
                "source_pair_idx": int(idx),
                "chunk_idx": i,
            }
        )

print(f"\nproduced {len(chunks)} chunks from {len(df)} response rows")
print(f"  mean words per chunk: {np.mean([len(c['text'].split()) for c in chunks]):.0f}")

chunking:   0%|          | 0/3512 [00:00<?, ?it/s]


produced 4030 chunks from 3512 response rows
  mean words per chunk: 161


## 5. Dense embedding with bge-m3

`BAAI/bge-m3` (1024-dim), embeddings normalized for cosine search in Qdrant. Batch size scales to the device.

In [6]:
EMBED_MODEL = "BAAI/bge-m3"
EMBED_DIM = 1024

embed_batch = {
    "hopper": 256,
    "ada": 128,
    "ampere": 256,
    "pre-ampere": 64,
    "none": 16,
}.get(GPU_ARCH, 16)

print(f"loading {EMBED_MODEL} on {DEVICE} (arch={GPU_ARCH}, batch={embed_batch})")
embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE)  # also used to embed queries at retrieval time
if BF16_OK:
    embedder.half()
    print("  using fp16/bf16 weights for inference")

texts = [c["text"] for c in chunks]

# Cache the corpus embeddings — bge-m3 on CPU is ~24 min over ~4k chunks, so we only pay it once.
EMB_CACHE = Path("/tmp/nb04_bge_m3_emb.npy")
embeddings = None
if EMB_CACHE.exists():
    cached = np.load(EMB_CACHE)
    if cached.shape == (len(texts), EMBED_DIM):
        embeddings = cached
        print(f"loaded cached embeddings {embeddings.shape} from {EMB_CACHE}")

if embeddings is None:
    t0 = time.perf_counter()
    embeddings = embedder.encode(
        texts, batch_size=embed_batch, normalize_embeddings=True,
        show_progress_bar=True, convert_to_numpy=True,
    )
    elapsed = time.perf_counter() - t0
    np.save(EMB_CACHE, embeddings)
    print(f"\nembedded {len(texts)} chunks in {elapsed:.1f}s ({len(texts) / elapsed:.1f} chunks/s); cached to {EMB_CACHE}")

print(f"embedding shape: {embeddings.shape}")
assert embeddings.shape == (len(texts), EMBED_DIM)

loading BAAI/bge-m3 on cpu (arch=none, batch=16)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/252 [00:00<?, ?it/s]


embedded 4030 chunks in 1378.9s (2.9 chunks/s); cached to /tmp/nb04_bge_m3_emb.npy
embedding shape: (4030, 1024)


## 6. Qdrant collection — idempotent recreate

In [7]:
QDRANT_COLLECTION = "sakina_counseling"

client = QdrantClient(
    url=os.environ["QDRANT_URL"],
    api_key=os.environ["QDRANT_API_KEY"],
    timeout=300,  # cloud writes can be slow
)

existing = [c.name for c in client.get_collections().collections]
print(f"existing collections: {existing}")

if QDRANT_COLLECTION in existing:
    info = client.get_collection(QDRANT_COLLECTION)
    print(f"  '{QDRANT_COLLECTION}' exists with {info.points_count} points — recreating")
    client.delete_collection(QDRANT_COLLECTION)

client.create_collection(
    collection_name=QDRANT_COLLECTION,
    vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE),
)
print(f"[OK] created '{QDRANT_COLLECTION}' (size={EMBED_DIM}, distance=COSINE)")

existing collections: ['sakina_counseling']


  'sakina_counseling' exists with 3584 points — recreating


[OK] created 'sakina_counseling' (size=1024, distance=COSINE)


## 7. Upsert points to Qdrant

Use integer point IDs (Qdrant prefers them). The `chunks[i]['id']` string is kept as a payload field.

In [8]:
UPSERT_BATCH = 64  # smaller batches + retries — cloud Qdrant timed out on 256-point batches


def _upsert_with_retry(points, tries: int = 4) -> None:
    for attempt in range(tries):
        try:
            client.upsert(collection_name=QDRANT_COLLECTION, points=points, wait=True)
            return
        except Exception as e:
            if attempt == tries - 1:
                raise
            print(f"  upsert retry {attempt + 1}/{tries} ({type(e).__name__}); backing off")
            time.sleep(5 * (attempt + 1))


t0 = time.perf_counter()
for start in tqdm(range(0, len(chunks), UPSERT_BATCH), desc="upserting"):
    end = start + UPSERT_BATCH
    batch_chunks = chunks[start:end]
    batch_embs = embeddings[start:end]
    points = [
        PointStruct(
            id=start + i,
            vector=emb.tolist(),
            payload={
                "chunk_id": c["id"],
                "text": c["text"],
                "context": c["context"],
                "source_pair_idx": c["source_pair_idx"],
                "chunk_idx": c["chunk_idx"],
            },
        )
        for i, (c, emb) in enumerate(zip(batch_chunks, batch_embs))
    ]
    _upsert_with_retry(points)
elapsed = time.perf_counter() - t0

info = client.get_collection(QDRANT_COLLECTION)
print(f"\n[OK] upserted in {elapsed:.1f}s; collection now has {info.points_count} points")
assert info.points_count == len(chunks), f"expected {len(chunks)} points, got {info.points_count}"

upserting:   0%|          | 0/63 [00:00<?, ?it/s]


[OK] upserted in 289.0s; collection now has 4030 points


## 8. BM25 sparse index

Pure-Python BM25 over the same corpus, persisted to `bm25_index.pkl` so the API loads it without re-tokenizing at startup. Tokenization is lowercase `\w+` (strong for English; dense retrieval covers other languages).

In [9]:
TOKEN_RE = re.compile(r"\w+", re.UNICODE)


def tokenize(text: str) -> list[str]:
    return TOKEN_RE.findall(text.lower())


tokenized_corpus = [tokenize(c["text"]) for c in chunks]
bm25 = BM25Okapi(tokenized_corpus)

print(f"BM25 index built over {len(tokenized_corpus)} docs")
print(f"  vocab size: {len(bm25.idf)}")
print(f"  avg doc length: {bm25.avgdl:.1f} tokens")

BM25 index built over 4030 docs
  vocab size: 14277
  avg doc length: 166.3 tokens


In [10]:
bm25_path = ARTIFACTS_DIR / "bm25_index.pkl"
with open(bm25_path, "wb") as f:
    pickle.dump(
        {
            "bm25": bm25,
            "chunk_ids": [c["id"] for c in chunks],
            "chunk_texts": [c["text"] for c in chunks],
            "chunk_contexts": [c["context"] for c in chunks],
            "metadata": {
                "version": "1.0",
                "trained_on": "Amod/mental_health_counseling_conversations",
                "n_chunks": len(chunks),
                "tokenizer": "lowercase + \\w+",
                "chunk_size_words": 300,
                "chunk_overlap_words": 50,
            },
        },
        f,
        protocol=pickle.HIGHEST_PROTOCOL,
    )
size_mb = bm25_path.stat().st_size / 1024 / 1024
print(f"[OK] saved {bm25_path} ({size_mb:.2f} MB)")

[OK] saved ../api/app/models/artifacts/bm25_index.pkl (8.66 MB)


## 9. Hybrid retrieval — BM25 + Dense + RRF fusion

Take top-10 from BM25 and top-10 from Qdrant, fuse via Reciprocal Rank Fusion (k=60), then rerank the fused top-10 with the cross-encoder and return top-3.

In [11]:
RRF_K = 60
TOP_K_PER_RETRIEVER = 10
TOP_K_RERANKED = 3

RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"
print(f"loading {RERANKER_MODEL} on {DEVICE}")
reranker = CrossEncoder(RERANKER_MODEL, device=DEVICE, max_length=512)
if BF16_OK:
    reranker.model = reranker.model.half()
    print("  using fp16/bf16 weights for inference")

loading BAAI/bge-reranker-v2-m3 on cpu


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

In [12]:
def bm25_topk(query: str, k: int = TOP_K_PER_RETRIEVER) -> list[tuple[int, float]]:
    """Return [(corpus_idx, score), ...] sorted high→low."""
    scores = bm25.get_scores(tokenize(query))
    top = np.argsort(scores)[::-1][:k]
    return [(int(i), float(scores[i])) for i in top if scores[i] > 0]


def dense_topk(query: str, k: int = TOP_K_PER_RETRIEVER) -> list[tuple[int, float]]:
    """Return [(qdrant_point_id, score), ...]. Point id == corpus index here."""
    q_emb = embedder.encode(query, normalize_embeddings=True, convert_to_numpy=True)
    hits = client.query_points(
        collection_name=QDRANT_COLLECTION,
        query=q_emb.tolist(),
        limit=k,
        with_payload=False,
    ).points
    return [(int(h.id), float(h.score)) for h in hits]


def rrf_fuse(
    ranked_lists: list[list[tuple[int, float]]], k: int = RRF_K
) -> list[tuple[int, float]]:
    """Reciprocal Rank Fusion. Returns [(doc_id, rrf_score), ...] sorted high→low."""
    scores: dict[int, float] = defaultdict(float)
    for ranked in ranked_lists:
        for rank, (doc_id, _) in enumerate(ranked, start=1):
            scores[doc_id] += 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


def rerank(query: str, doc_ids: list[int], k: int = TOP_K_RERANKED) -> list[tuple[int, float]]:
    """Cross-encoder rerank. Returns final top-k."""
    pairs = [(query, chunks[i]["text"]) for i in doc_ids]
    scores = reranker.predict(pairs, show_progress_bar=False)
    order = np.argsort(scores)[::-1][:k]
    return [(doc_ids[i], float(scores[i])) for i in order]


def hybrid_retrieve(query: str, k: int = TOP_K_RERANKED) -> list[dict]:
    """Full pipeline: BM25 + Dense → RRF → cross-encoder rerank → top-k chunks."""
    bm25_hits = bm25_topk(query)
    dense_hits = dense_topk(query)
    fused = rrf_fuse([bm25_hits, dense_hits])
    fused_ids = [doc_id for doc_id, _ in fused[:TOP_K_PER_RETRIEVER]]
    reranked = rerank(query, fused_ids, k=k)
    return [
        {
            "doc_id": doc_id,
            "score": score,
            "text": chunks[doc_id]["text"],
            "context": chunks[doc_id]["context"],
        }
        for doc_id, score in reranked
    ]

## 10. Sanity check — sample query → top-3 reranked

In [13]:
SAMPLE_QUERY = "I can't sleep at night because I keep worrying about everything. What can I do?"

print(f"Q: {SAMPLE_QUERY}\n")
results = hybrid_retrieve(SAMPLE_QUERY, k=3)
for i, r in enumerate(results, 1):
    print(f"--- rank {i}  (rerank score = {r['score']:.3f}) ---")
    print(f"context: {r['context'][:200]}...")
    print(f"text   : {r['text'][:400]}...")
    print()

Q: I can't sleep at night because I keep worrying about everything. What can I do?



--- rank 1  (rerank score = 0.806) ---
context: I have been having a lot of nightmares where I am being killed in different ways. I either wake up in a panic or just crying and sweating. It has made me terrified of falling asleep and is now affecti...
text   : Hello, and thank you for your question. Sleep problems, including insomnia and even the nightmares that you are writing about, are really common for people and so many people suffer from them. Here are a few things to consider or to try: 1. Have you changed or started taking any new medication lately? Medications can certainly affect your sleep and some can even cause vivid or frightening dreams. ...

--- rank 2  (rerank score = 0.806) ---
context: I have been having a lot of nightmares where I am being killed in different ways. I either wake up in a panic or just crying and sweating. It has made me terrified of falling asleep and is now affecti...
text   : Hello, and thank you for your question. Sleep problems, including insomni

## 11. Retrieval evaluation — Hit@5 + MRR@5

Hand-crafted probe set (query + a substring the correct chunk must contain). Hit@5 = fraction of queries where a top-5 fused (pre-rerank) hit contains the substring; MRR@5 = mean `1/rank` of the first match. A smoke test, not a held-out benchmark.

In [14]:
PROBE_SET = [
    # (query, expected substring that should appear in a relevant chunk)
    ("I feel anxious all the time and can't relax", "anxiety"),
    ("I'm having trouble sleeping because of stress", "sleep"),
    ("My partner and I keep arguing about small things", "relationship"),
    ("I think I might be depressed", "depress"),
    ("I don't feel motivated to do anything anymore", "motivat"),
    ("How do I deal with grief after losing someone", "grief"),
    ("My self-esteem is really low", "self-esteem"),
    ("I have panic attacks at work", "panic"),
    ("I feel lonely even when surrounded by people", "lonely"),
    ("How can I stop overthinking everything", "overthink"),
]


def hit_and_rank(query: str, expected: str, k: int = 5) -> tuple[bool, int | None]:
    bm25_hits = bm25_topk(query)
    dense_hits = dense_topk(query)
    fused = rrf_fuse([bm25_hits, dense_hits])[:k]
    for rank, (doc_id, _) in enumerate(fused, start=1):
        if expected.lower() in chunks[doc_id]["text"].lower():
            return True, rank
    return False, None


hits = 0
rr_sum = 0.0
rows = []
for q, expected in tqdm(PROBE_SET, desc="eval"):
    hit, rank = hit_and_rank(q, expected, k=5)
    hits += int(hit)
    rr_sum += (1.0 / rank) if hit else 0.0
    rows.append({"query": q[:50] + "...", "expected": expected, "hit@5": hit, "rank": rank})

hit_at_5 = hits / len(PROBE_SET)
mrr_at_5 = rr_sum / len(PROBE_SET)

print(pd.DataFrame(rows).to_string(index=False))
print(f"\nHit@5 = {hit_at_5:.3f}  (DoD: > 0.85)")
print(f"MRR@5 = {mrr_at_5:.3f}  (target: > 0.75 per pipeline doc §10)")

eval:   0%|          | 0/10 [00:00<?, ?it/s]

                                              query     expected  hit@5  rank
     I feel anxious all the time and can't relax...      anxiety   True   1.0
   I'm having trouble sleeping because of stress...        sleep   True   1.0
My partner and I keep arguing about small things... relationship   True   1.0
                    I think I might be depressed...      depress   True   1.0
   I don't feel motivated to do anything anymore...      motivat   True   2.0
   How do I deal with grief after losing someone...        grief   True   1.0
                    My self-esteem is really low...  self-esteem   True   1.0
                    I have panic attacks at work...        panic   True   1.0
    I feel lonely even when surrounded by people...       lonely  False   NaN
          How can I stop overthinking everything...    overthink  False   NaN

Hit@5 = 0.800  (DoD: > 0.85)
MRR@5 = 0.750  (target: > 0.75 per pipeline doc §10)


## 12. Generation-quality eval — LLM-as-judge

A judge LLM scores each (query, retrieved-context, answer) on **faithfulness** (claims supported by context) and **answer_relevancy** (does it answer the question), 0–1. Dependency-free replacement for RAGAS, using the same Lightning client.

In [15]:
from openai import OpenAI

llm_client = OpenAI(
    base_url=os.environ["LIGHTNING_BASE_URL"],
    api_key=os.environ["LIGHTNING_API_KEY"],
)
LIGHTNING_MODEL = os.environ.get("LIGHTNING_MODEL", "lightning-ai/gpt-oss-120b")
rl = LightningRateLimiter()


def llm_answer(query: str, contexts: list[str]) -> str:
    """Synthesize a grounded answer from retrieved contexts. Used for RAGAS."""
    ctx_block = "\n\n".join(f"[{i + 1}] {c}" for i, c in enumerate(contexts))
    prompt = (
        "You are a warm, careful mental-health support assistant. Use the retrieved "
        "counselor notes below to compose a brief (2-4 sentence) empathetic reply to the "
        "user. Ground every claim in the notes. If the notes don't address the question, "
        "say so plainly.\n\n"
        f"--- RETRIEVED NOTES ---\n{ctx_block}\n--- END NOTES ---\n\n"
        f"User: {query}\n\nReply:"
    )
    rl.acquire(estimated_tokens=len(prompt.split()) + 200)
    resp = llm_client.chat.completions.create(
        model=LIGHTNING_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=300,
        temperature=0.3,
    )
    return resp.choices[0].message.content.strip()


# Pick the first 3 probe queries to keep this cheap
ragas_eval_queries = [q for q, _ in PROBE_SET[:3]]
ragas_rows = []
for q in ragas_eval_queries:
    retrieved = hybrid_retrieve(q, k=3)
    contexts = [r["text"] for r in retrieved]
    answer = llm_answer(q, contexts)
    ragas_rows.append(
        {
            "question": q,
            "contexts": contexts,
            "answer": answer,
        }
    )
    print(f"Q: {q}")
    print(f"A: {answer[:200]}...\n")

Q: I feel anxious all the time and can't relax
A: I’m really sorry you’re feeling anxious all the time—it can be exhausting and make relaxation feel impossible. One practical first step is to start tracking what seems to trigger each episode, and if ...



Q: I'm having trouble sleeping because of stress
A: I’m really sorry you’re dealing with sleepless nights—stress is a common trigger for the kind of sleep trouble you’re describing, as the notes explain. It can help to pinpoint the specific stressors i...



Q: My partner and I keep arguing about small things
A: I hear how frustrating it can feel when the disagreements seem to center on the little, everyday details. As the notes point out, arguments over minor issues often stem from temporary stress or habits...



In [16]:
import json as _json

import pandas as pd


def _judge(query: str, contexts: list[str], answer: str) -> tuple[float, float]:
    ctx = "\n\n".join(f"[{i + 1}] {c}" for i, c in enumerate(contexts))
    prompt = (
        "You are scoring a mental-health assistant reply on two axes, each 0.0-1.0:\n"
        "- faithfulness: is every claim in the reply supported by the CONTEXT notes?\n"
        "- answer_relevancy: does the reply actually address the USER question?\n\n"
        f"USER: {query}\n\nCONTEXT:\n{ctx}\n\nASSISTANT REPLY:\n{answer}\n\n"
        'Output ONLY JSON: {"faithfulness": <float>, "answer_relevancy": <float>}'
    )
    rl.acquire(estimated_tokens=len(prompt.split()) + 60)
    r = llm_client.chat.completions.create(
        model=LIGHTNING_MODEL, temperature=0.0, max_tokens=200,
        messages=[{"role": "user", "content": prompt}],
    )
    out = r.choices[0].message.content or ""
    a, b = out.find("{"), out.rfind("}")
    try:
        d = _json.loads(out[a:b + 1])
        return float(d.get("faithfulness", 0.0)), float(d.get("answer_relevancy", 0.0))
    except Exception:
        return 0.0, 0.0


judged = []
for row in ragas_rows:
    f, rel = _judge(row["question"], row["contexts"], row["answer"])
    judged.append({"question": row["question"][:50] + "...", "faithfulness": f, "answer_relevancy": rel})

jdf = pd.DataFrame(judged)
print(jdf.to_string(index=False))
faith_mean = float(jdf["faithfulness"].mean())
rel_mean = float(jdf["answer_relevancy"].mean())
print(f"\nfaithfulness     = {faith_mean:.3f}  (DoD: > 0.80)")
print(f"answer_relevancy = {rel_mean:.3f}  (target: > 0.78 per pipeline doc §10)")

                                           question  faithfulness  answer_relevancy
     I feel anxious all the time and can't relax...           1.0               1.0
   I'm having trouble sleeping because of stress...           1.0               1.0
My partner and I keep arguing about small things...           1.0               1.0

faithfulness     = 1.000  (DoD: > 0.80)
answer_relevancy = 1.000  (target: > 0.78 per pipeline doc §10)


## 13. Summary

In [17]:
summary = {
    "corpus": {
        "dataset": "Amod/mental_health_counseling_conversations",
        "pairs": len(df),
        "chunks": len(chunks),
        "chunk_size_words": 300,
        "chunk_overlap_words": 50,
    },
    "dense": {
        "model": EMBED_MODEL,
        "dim": EMBED_DIM,
        "qdrant_collection": QDRANT_COLLECTION,
        "qdrant_points": client.get_collection(QDRANT_COLLECTION).points_count,
    },
    "sparse": {
        "model": "BM25Okapi (rank-bm25)",
        "vocab_size": len(bm25.idf),
        "artifact": str(bm25_path),
    },
    "hybrid": {
        "fusion": f"RRF (k={RRF_K})",
        "top_k_per_retriever": TOP_K_PER_RETRIEVER,
    },
    "rerank": {
        "model": RERANKER_MODEL,
        "top_k_final": TOP_K_RERANKED,
    },
    "eval": {
        "hit_at_5": float(hit_at_5),
        "mrr_at_5": float(mrr_at_5),
        "ragas_faithfulness": float(faith_mean),
        "ragas_answer_relevancy": float(rel_mean),
    },
}
print(json.dumps(summary, indent=2))

{
  "corpus": {
    "dataset": "Amod/mental_health_counseling_conversations",
    "pairs": 3512,
    "chunks": 4030,
    "chunk_size_words": 300,
    "chunk_overlap_words": 50
  },
  "dense": {
    "model": "BAAI/bge-m3",
    "dim": 1024,
    "qdrant_collection": "sakina_counseling",
    "qdrant_points": 4030
  },
  "sparse": {
    "model": "BM25Okapi (rank-bm25)",
    "vocab_size": 14277,
    "artifact": "../api/app/models/artifacts/bm25_index.pkl"
  },
  "hybrid": {
    "fusion": "RRF (k=60)",
    "top_k_per_retriever": 10
  },
  "rerank": {
    "model": "BAAI/bge-reranker-v2-m3",
    "top_k_final": 3
  },
  "eval": {
    "hit_at_5": 0.8,
    "mrr_at_5": 0.75,
    "ragas_faithfulness": 1.0,
    "ragas_answer_relevancy": 1.0
  }
}
